# Three factorizations and a curve fit

To solve Ax = b, factor A once, then solve with the factors. Which
factorization depends on what A is. LU works for any nonsingular matrix,
Cholesky only for a symmetric positive definite one, and QR also solves
least-squares problems, where A is tall. This notebook solves one system all
ways, shows on an ill-conditioned matrix why the inverse is the one to
avoid, shows Cholesky refusing a matrix it can't factor, and then fits a
curve, where the choice between QR and the normal equations decides how many
digits survive. The matrices come from MTL5's test-matrix generators.

**Uses:** `mtl5.lu`, `mtl5.cholesky`, `mtl5.qr`, `mtl5.ldlt`,
`mtl5.bunch_kaufman`, `mtl5.inv`, `mtl5.inertia`, `mtl5.is_spd`,
`mtl5.condition_number`, `mtl5.convert`, and `mtl5.generators` (`lehmer`,
`hilbert`, `wilkinson`).

**From C++:** [`phase05_dense_decompositions/solve_three_ways.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/solve_three_ways.cpp), [`phase05_dense_decompositions/least_squares_qr.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/least_squares_qr.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## One system, three factorizations

MTL5's test-matrix generators, `mtl5.generators`, give matrices whose
properties are known. The Lehmer matrix, A[i, j] = min(i, j) / max(i, j)
(counting from 1), is symmetric positive definite and well conditioned, and
its inverse is tridiagonal. All three factorizations apply. The right-hand
side is b = A x with x = (1, 2, …, n), so the error of each solution is known.

Each factorization returns an object that keeps the factors and solves with
them. Factor once, and each further right-hand side costs only the solve.

In [ ]:
n = 6
A = mtl5.generators.lehmer(n).to_numpy()
x_true = np.arange(1.0, n + 1.0)
b = A @ x_true

assert mtl5.is_spd(mtl5.matrix(A))
assert np.allclose(np.triu(np.linalg.inv(A), 2), 0.0, atol=1e-12)  # tridiagonal inverse

solutions = {
    "LU": mtl5.lu(A).solve(mtl5.vector(b)).to_numpy(),
    "Cholesky": mtl5.cholesky(A).solve(mtl5.vector(b)).to_numpy(),
    "QR": mtl5.qr(A).solve(mtl5.vector(b)).to_numpy(),
    "inv(A) @ b": mtl5.inv(A).to_numpy() @ b,
}
eps = np.finfo(float).eps
kappa = mtl5.condition_number(mtl5.matrix(A))
print(f"cond(A) = {kappa:.1f}\n")
for name, x in solutions.items():
    error = np.linalg.norm(x - x_true) / np.linalg.norm(x_true)
    print(f"{name:>11}: relative error {error:.1e}")
    assert error <= 10 * kappa * eps, name

On a well-conditioned matrix all four answers are right to the last digit or
two, so this system can't tell the methods apart. An ill-conditioned one can.

## Where they differ: the Hilbert matrix

The Hilbert matrix, H[i, j] = 1 / (i + j + 1) (counting from 0), is symmetric
positive definite and notoriously ill-conditioned: cond(H) grows about 30-fold
with each row. Solve H x = b for n = 4 to 15, with x all ones, and measure two
errors of each computed x̂:

- the **forward error** ‖x̂ − x‖ / ‖x‖, how wrong the answer is;
- the **backward error** ‖b − H x̂‖ / (‖H‖ ‖x̂‖), the smallest relative change
  to H for which x̂ is the exact answer.

A backward-stable method keeps the backward error near ε however badly H is
conditioned. Its forward error is then at most about cond(H) · ε, which is
the best any method can do once H is rounded to double precision.

In [ ]:
methods = {
    "LU": lambda H, b: mtl5.lu(H).solve(mtl5.vector(b)).to_numpy(),
    "Cholesky": lambda H, b: mtl5.cholesky(H).solve(mtl5.vector(b)).to_numpy(),
    "QR": lambda H, b: mtl5.qr(H).solve(mtl5.vector(b)).to_numpy(),
    "inv(A) @ b": lambda H, b: mtl5.inv(H).to_numpy() @ b,
}
sizes = np.arange(4, 16)
cond_H = []
forward = {name: [] for name in methods}
backward = {name: [] for name in methods}
for n in sizes:
    H = mtl5.generators.hilbert(n).to_numpy()
    x = np.ones(n)
    b = H @ x
    cond_H.append(mtl5.condition_number(mtl5.matrix(H)))
    for name, solve in methods.items():
        try:
            xh = solve(H, b)
        except RuntimeError:
            if name != "Cholesky":
                raise  # only Cholesky may refuse: H is no longer SPD in floating point
            forward[name].append(np.nan)
            backward[name].append(np.nan)
            continue
        forward[name].append(np.linalg.norm(xh - x) / np.linalg.norm(x))
        scale = np.linalg.norm(H, 2) * np.linalg.norm(xh)
        backward[name].append(np.linalg.norm(b - H @ xh) / scale)
cond_H = np.array(cond_H)
forward = {k: np.array(v) for k, v in forward.items()}
backward = {k: np.array(v) for k, v in backward.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.0, 3.8), sharex=True)
markers = {"LU": "o", "Cholesky": "s", "QR": "^", "inv(A) @ b": "D"}
for color, name in zip(SERIES, methods):
    ax1.semilogy(sizes, forward[name], markers[name] + "-", color=color, label=name, markersize=4)
    # an exact solve has backward error 0, which a log axis can't show: leave it out
    shown = np.where(backward[name] > 0, backward[name], np.nan)
    ax2.semilogy(sizes, shown, markers[name] + "-", color=color, label=name, markersize=4)
ax1.semilogy(sizes, cond_H * eps, ":", color=INK_2, label="cond(H) · ε")
ax2.axhline(eps, color=INK_2, linestyle=":", label="ε")
ax1.set_title("Forward error ‖x̂ − x‖ / ‖x‖", fontsize=10)
ax2.set_title("Backward error ‖b − Hx̂‖ / (‖H‖ ‖x̂‖)", fontsize=10)
for ax in (ax1, ax2):
    ax.set_xlabel("n (Hilbert matrix n × n)")
    ax.grid(True)
ax1.legend(frameon=False, fontsize=8)
ax2.legend(handles=ax2.get_lines()[-1:], frameon=False, fontsize=8, loc="center right")
plt.tight_layout()
plt.show()

first_failure = sizes[np.isnan(forward["Cholesky"])]
print("Cholesky first fails at n =", first_failure[0] if first_failure.size else "never")

LU, Cholesky and QR are backward stable: their backward error stays at the
level of ε at every size, and their forward error tracks cond(H) · ε.
`inv(A) @ b` is not. Its backward error grows with cond(H), many orders of
magnitude above the others, and by n ≈ 10 its forward error falls behind
too. So the inverse costs accuracy as well as work. Forming A⁻¹ takes about
2n³ operations, LU about (2/3)n³ and Cholesky half that, and a sparse matrix
has a dense inverse, which can't even be stored for the matrices of the later
examples.

Cholesky stops at the largest sizes. There cond(H) is far past 1/ε ≈ 4.5e15,
and the rounded H has a pivot that isn't positive. The checks:

In [ ]:
# Past n = 12, cond(H) exceeds 1/eps and the computed value is itself rounding noise.
well_posed = sizes <= 12
# LU, QR and the inverse solve at every size; only Cholesky may have stopped
for name in ("LU", "QR", "inv(A) @ b"):
    assert np.isfinite(forward[name]).all() and np.isfinite(backward[name]).all(), name
for name in ("LU", "Cholesky", "QR"):
    ok = ~np.isnan(backward[name])
    # backward stable: within a small multiple of eps, at every size
    assert (backward[name][ok] <= 10 * eps).all(), name
    # and so accurate to about cond(H) * eps
    assert (forward[name][well_posed & ok] <= 10 * cond_H[well_posed & ok] * eps).all(), name
assert not np.isnan(forward["Cholesky"][sizes <= 10]).any()  # Cholesky works while H is safely SPD

# the inverse's backward error is at least 100x the stable methods', from n = 6 on
stable_worst = np.nanmax([backward[m] for m in ("LU", "Cholesky", "QR")], axis=0)
assert (backward["inv(A) @ b"][sizes >= 6] >= 100 * stable_worst[sizes >= 6]).all()
print("LU, Cholesky, QR backward stable; inv(A) @ b at least 100x worse from n = 6")

## A matrix Cholesky can't factor

The Wilkinson matrix W₇⁺ is symmetric and tridiagonal, with 3, 2, 1, 0, 1, 2, 3
on the diagonal and ones beside it. One of its eigenvalues is negative, so it
is not positive definite. Cholesky needs a square root of each pivot, and
stops with an error when a pivot isn't positive. LU and QR need neither
symmetry nor definiteness, and still solve.

In [ ]:
W = mtl5.generators.wilkinson(7)
A_indef = W.to_numpy()
x_true = np.arange(1.0, 8.0)
b_indef = A_indef @ x_true
print("symmetric:", mtl5.is_symmetric(W), " SPD:", mtl5.is_spd(W))

try:
    mtl5.cholesky(A_indef)
except RuntimeError as err:
    print("Cholesky:", err)
else:
    raise AssertionError("Cholesky should reject an indefinite matrix")

for name, factor in [("LU", mtl5.lu), ("QR", mtl5.qr)]:
    x = factor(A_indef).solve(mtl5.vector(b_indef)).to_numpy()
    print(f"{name}: error {np.linalg.norm(x - x_true):.1e}")
    assert np.allclose(x, x_true, rtol=0, atol=1e-13)

For a symmetric indefinite matrix there is a factorization in between:
LDLᵀ takes no square roots, so negative pivots are allowed. The signs of D
are the matrix's inertia, its count of positive and negative eigenvalues
(Sylvester's law of inertia). So the factorization also tells how far from
definite the matrix is: one eigenvalue of W₇⁺ is negative.

In [ ]:
F = mtl5.ldlt(A_indef)
D = np.asarray(F.diagonal())
x = F.solve(mtl5.vector(b_indef)).to_numpy()
counts = mtl5.inertia(W)
print("D =", np.round(D, 4), "  inertia:", counts)

assert np.allclose(x, x_true, rtol=0, atol=1e-13)
assert (D < 0).sum() == counts["negative"] == 1
assert (D > 0).sum() == counts["positive"] == 6

`ldlt` doesn't pivot, so it fails on a zero pivot even when the matrix is
nonsingular. The smallest example is [[0, 1], [1, 0]]. Bunch–Kaufman is LDLᵀ
with symmetric pivoting, using 2 × 2 blocks in D where a 1 × 1 pivot would be
zero or too small, and it solves that matrix.

In [ ]:
swap = np.array([[0.0, 1.0], [1.0, 0.0]])
try:
    mtl5.ldlt(swap)
except RuntimeError as err:
    print(str(err).split(";")[0])
else:
    raise AssertionError("ldlt should stop on the zero pivot")

x = mtl5.bunch_kaufman(swap).solve(mtl5.vector(np.array([2.0, 3.0]))).to_numpy()
print("bunch_kaufman: x =", x)
assert np.array_equal(x, [3.0, 2.0])

Which factorization, then:

| A is | Use | Python |
| --- | --- | --- |
| symmetric positive definite | Cholesky, the cheapest | `mtl5.cholesky(A)` |
| symmetric indefinite | LDLᵀ with pivoting | `mtl5.bunch_kaufman(A)`; `mtl5.ldlt(A)` doesn't pivot, and raises on a zero pivot |
| square, nothing more known | LU with partial pivoting | `mtl5.lu(A)`, or `mtl5.solve(A, b)` for one right-hand side |
| tall, a least-squares problem | QR | `mtl5.qr(A)` |

## Fitting a polynomial: QR against the normal equations

Fit a polynomial of degree p to 20 samples of sin x on [0, π], with the
tutorial's deterministic noise. The coefficients c minimize ‖Vc − y‖, where V
is the Vandermonde matrix with V[i, k] = xᵢᵏ. V is 20 × (p + 1), tall, so
it comes from `np.vander`: `mtl5.generators.vandermonde(nodes)` builds the
square Vandermonde matrix, one column per node, for interpolation rather
than fitting.

There are two ways to solve it. The normal equations VᵀV c = Vᵀy are a small
SPD system, so Cholesky solves them. QR works on V itself. The difference is
the condition number: cond(VᵀV) = cond(V)², and the Vandermonde matrix's
condition number grows exponentially with the degree.

In [ ]:
m = 20
k = np.arange(m)
x_data = np.pi * k / (m - 1)
y_data = np.sin(x_data) + 0.02 * np.sin(7.0 * k + 3.0)


def vandermonde(p):
    return np.vander(x_data, p + 1, increasing=True)


def fit_qr(V):
    return mtl5.qr(V).solve(mtl5.vector(y_data)).to_numpy()


def fit_normal(V):
    # raises RuntimeError once rounding leaves V^T V not positive definite
    return mtl5.cholesky(V.T @ V).solve(mtl5.vector(V.T @ y_data)).to_numpy()


fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(x_data, y_data, "o", color=INK_2, markersize=4, label="data")
fine = np.linspace(0.0, np.pi, 300)
for color, p in zip(SERIES, [1, 3, 8]):
    c = fit_qr(vandermonde(p))
    ax.plot(fine, np.polynomial.polynomial.polyval(fine, c), color=color, label=f"degree {p}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Least-squares polynomial fits to noisy sin x")
ax.legend(frameon=False)
ax.grid(True)
plt.tight_layout()
plt.show()

The plot can't show which method is more accurate: at these degrees the
fitted curves would lie on top of each other. The coefficients show it. To
measure their error, a reference solution is needed that is more accurate
than double precision. `mtl5.convert` puts V and y in quad-double, about 64
decimal digits, and the same `mtl5.qr` solves there. Rounded back to double,
it is the exact least-squares solution for this data, to double precision.

In [ ]:
degrees = np.arange(2, 14)
cond_V, err_qr, err_normal = [], [], []
for p in degrees:
    V = vandermonde(p)
    ref = mtl5.qr(mtl5.convert(V, "qd_cascade")).solve(mtl5.convert(y_data, "qd_cascade"))
    ref = np.asarray(ref.to_numpy(), dtype=float)

    def rel(c):
        return np.linalg.norm(c - ref) / np.linalg.norm(ref)

    cond_V.append(mtl5.condition_number(mtl5.matrix(V)))
    err_qr.append(rel(fit_qr(V)))
    try:
        err_normal.append(rel(fit_normal(V)))
    except RuntimeError:
        err_normal.append(np.nan)  # Cholesky failed: V^T V is not SPD in floating point
cond_V, err_qr, err_normal = map(np.array, (cond_V, err_qr, err_normal))

fig, ax = plt.subplots(figsize=(5.8, 3.8))
ax.semilogy(degrees, err_normal, "s-", color=SERIES[1], label="normal equations (Cholesky)")
ax.semilogy(degrees, err_qr, "o-", color=SERIES[0], label="QR")
ax.semilogy(degrees, cond_V**2 * eps, ":", color=SERIES[1], label="cond(V)² · ε")
ax.semilogy(degrees, cond_V * eps, ":", color=SERIES[0], label="cond(V) · ε")
ax.axhline(1.0, color=INK_2, linewidth=0.8)
ax.set_xlabel("polynomial degree p")
ax.set_ylabel("relative error of the coefficients")
ax.set_title("Error of the fitted coefficients, against quad-double")
ax.set_ylim(1e-16, 10.0)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

for p, kv, eq, en in zip(degrees, cond_V, err_qr, err_normal):
    print(f"p = {p:2d}  cond(V) = {kv:8.1e}  QR {eq:8.1e}  normal equations {en:8.1e}")

The rule of thumb holds: if cond(V) ≈ 10ᵏ, QR loses about k digits and the
normal equations about 2k. By degree 10 the normal equations have one or two
digits left, while QR still has nine. Past cond(V)² ≈ 1/ε, the normal
equations stop being definite in floating point, and Cholesky stops (the
missing points). The checks are on the bounds, not on the exact errors:

In [ ]:
# QR is backward stable: its error stays within a small multiple of cond(V) * eps
assert (err_qr <= 10 * cond_V * eps).all()

# from degree 6 on, the normal equations lose at least 100x more than QR,
# or fail outright
worse = degrees >= 6
assert (np.isnan(err_normal[worse]) | (err_normal[worse] >= 1e2 * err_qr[worse])).all()
print("QR within 10 cond(V) eps; normal equations at least 100x worse from degree 6")

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `lu_factor(A, pivot)`, `lu_solve(A, pivot, x, b)` | `F = mtl5.lu(A)`, `F.solve(b)` |
| `cholesky_factor(A)`, `cholesky_solve(A, x, b)` | `mtl5.cholesky(A).solve(b)` |
| `cholesky_factor(A)` returns the failing step | `mtl5.cholesky(A)` raises `RuntimeError` |
| `qr_factor(A, tau)`, `qr_solve(A, tau, x, b)` | `mtl5.qr(A).solve(b)`; `.Q` and `.R` are available |
| `inv(A) * b` | `mtl5.inv(A)`, best avoided |
| `trans(V) * V` | `V.T @ V` on the NumPy array |
| `eigenvalue_symmetric(VtV)`, then the ratio of extremes | `mtl5.condition_number(V)`, from the SVD |
| `dense2D<qd>` | `mtl5.convert(V, "qd_cascade")`, the same `mtl5.qr` |
| a matrix built entry by entry | `mtl5.generators.lehmer(n)`, `hilbert(n)`, `wilkinson(n)` |

The C++ factorizations overwrite A with the factors, so the program copies A
before each one. The Python ones leave A as it is.

The C++ program builds its own test matrices: 1 / (1 + |i − j|) plus the
identity, and the same with two diagonal entries lowered to 0.1. The first is
well conditioned (cond ≈ 2.3 at n = 4), so every method agrees on it and it
can't show why one is preferred. This notebook uses MTL5's generators instead,
whose properties are documented: Lehmer for the easy system, Hilbert for the
hard one, and Wilkinson's W₇⁺ for the indefinite one.

Part 3 of [`phase05_dense_decompositions/solve_three_ways.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/solve_three_ways.cpp), ILU(0) and IC(0) as preconditioners, compares
iteration counts. It moves to the Krylov solvers notebook, once the Python
solvers report iterations (#147).